In [25]:
import torch
import numpy as np
from collections import deque
from model import EventTCN, frame_to_feature_vector
from ultralytics import YOLO
import cv2


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)
K = 15
WINDOW = 16   # number of timesteps for TCN
STRIDE = 1    # step between windows
NUM_FEATURES = 4
text_tension = ''
# Define the font, scale, color, and thickness
font = cv2.FONT_HERSHEY_SIMPLEX
font_scale = 2
color = (0, 255, 0)  # Green color in BGR format
thickness = 2
# 1. Load model
det_model = YOLO("yolo12m.pt")
ckpt = torch.load("tcn_best_by_loss.pt", weights_only=True, map_location=DEVICE)
#print(ckpt)
model = EventTCN(input_dim=61, hidden_dim=64, num_layers=3).to(DEVICE)
model.load_state_dict(ckpt)
model.eval()

window_size = 16
threshold = 0.7
score = 0
buffer = deque(maxlen=window_size)
prev_pos = False
path = '/home/anna/Videos/video/6_11/7.mp4'
cap = cv2.VideoCapture(path)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video {path}")
print(cap)
output_filename = 'output_video_6_7.avi'  # Name of the output video file
fourcc = cv2.VideoWriter_fourcc(*'XVID')  # Codec (e.g., 'XVID', 'MJPG', 'MP4V')
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) 
frame_idx = 0
out = cv2.VideoWriter(output_filename, fourcc, fps, (width, height))

while True:
    ret, frame = cap.read()
    if not ret:
        break
    #print(frame_idx, frame.shape)
    # If you already sample every 5th frame, you can filter here:
    if frame_idx % 5 != 0:
        cv2.putText(frame, text_tension , (100,100), font, font_scale, color, thickness, cv2.LINE_AA)
        cv2.imshow("tension", frame)
        frame_idx += 1
        out.write(frame)
        continue
    results = det_model(frame, verbose=False)[0]         # for group event selection: list of (cls, conf)
    bbs = []             # for JSON output: [cls, conf, x1, y1, x2, y2]
    #group_events = []
    #individual_events = []
    
    #print(results.boxes, results.keypoints.xyn, results.keypoints.conf)
    if results.boxes:
        for box in results.boxes:#, results.keypoints.xyn, results.keypoints.conf):
            cls_id = int(box.cls)
            conf = float(box.conf)
            if conf < 0.5:
                continue
            x1, y1, x2, y2 = map(float, box.xyxyn[0])
            bbs.append([x1, y1, x2, y2])    
    # 1) Extract your per-frame features (cx, cy, w, h ...):
    feat = frame_to_feature_vector(bbs, K, NUM_FEATURES)   # np.array (C,)

    buffer.append(feat)
    frame_idx += 1

    # 2) When we have enough frames and we're on a stride boundary:
    if len(buffer) == WINDOW:# and (frame_idx - 1) % STRIDE == (WINDOW - 1) % STRIDE:
    # buffer is an ordered window of 16 feature vectors
        X_win = np.stack(buffer, axis=0).astype(np.float32)  # (16, C)
        X_tensor = torch.from_numpy(X_win).unsqueeze(0).to(DEVICE)  # (1, 16, 451)
    #yield frame_idx - WINDOW, X_win  # start index, window
        with torch.no_grad():
            logit = model(X_tensor)          # (1,)
            score = torch.sigmoid(logit)[0].item()
    
        is_pos = score >= threshold
    
        # 4. simple smoothing: require 2 consecutive positives
        if is_pos and prev_pos:
            text_tension = f"TENSION detected (score={score:.3f})"
            color = (0, 0, 255) 
            #print(text_tension)
        else:
            text_tension = 'NO'
            color = (0, 255, 0) 
        prev_pos = is_pos
        
    cv2.putText(frame, text_tension , (100,100), font, font_scale, color, thickness, cv2.LINE_AA)
    cv2.imshow("tension", frame)
    out.write(frame)
    key = cv2.waitKey(1) & 0xFF
    if key == 27:   # ESC to quit
        break
cap.release()
out.release()
cv2.destroyAllWindows()

cuda
< cv2.VideoCapture 0x73d59c55c890>


[h264 @ 0x5d5329793b80] error while decoding MB 66 23, bytestream -34


In [1]:
print('no')

no


In [ ]:
import cv2
from collections import deque
import numpy as np
import torch
from model import EventTCN, frame_to_vector

WINDOW = 16   # number of timesteps for TCN
STRIDE = 1    # step between windows

def iter_windows_from_video(path):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video {path}")

    buffer = deque(maxlen=WINDOW)
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # If you already sample every 5th frame, you can filter here:
        # if frame_idx % 5 != 0:
        #     frame_idx += 1
        #     continue

        # 1) Extract your per-frame features (cx, cy, w, h ...):
        feat = build_feature_vector_from_frame(frame)   # np.array (C,)

        buffer.append(feat)
        frame_idx += 1

        # 2) When we have enough frames and we're on a stride boundary:
        if len(buffer) == WINDOW and (frame_idx - 1) % STRIDE == (WINDOW - 1) % STRIDE:
            # buffer is an ordered window of 16 feature vectors
            X_win = np.stack(buffer, axis=0).astype(np.float32)  # (16, C)
            yield frame_idx - WINDOW, X_win  # start index, window

    cap.release()
